In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
import warnings
warnings.filterwarnings('ignore')

In [4]:
DATA = './data'

In [5]:
# =============================================================================
# 1. LOAD DATA
# =============================================================================
print("=" * 60)
print("STEP 1: Loading data...")
print("=" * 60)

m_teams = pd.read_csv(f'{DATA}/MTeams.csv')
w_teams = pd.read_csv(f'{DATA}/WTeams.csv')

m_reg = pd.read_csv(f'{DATA}/MRegularSeasonCompactResults.csv')
w_reg = pd.read_csv(f'{DATA}/WRegularSeasonCompactResults.csv')

m_tourney = pd.read_csv(f'{DATA}/MNCAATourneyCompactResults.csv')
w_tourney = pd.read_csv(f'{DATA}/WNCAATourneyCompactResults.csv')

m_seeds = pd.read_csv(f'{DATA}/MNCAATourneySeeds.csv')
w_seeds = pd.read_csv(f'{DATA}/WNCAATourneySeeds.csv')

m_detail = pd.read_csv(f'{DATA}/MRegularSeasonDetailedResults.csv')
w_detail = pd.read_csv(f'{DATA}/WRegularSeasonDetailedResults.csv')

m_tourney_detail = pd.read_csv(f'{DATA}/MNCAATourneyDetailedResults.csv')
w_tourney_detail = pd.read_csv(f'{DATA}/WNCAATourneyDetailedResults.csv')

m_conf = pd.read_csv(f'{DATA}/MTeamConferences.csv')
w_conf = pd.read_csv(f'{DATA}/WTeamConferences.csv')

submission = pd.read_csv(f'{DATA}/SampleSubmissionStage2.csv')

print(f"  Men's regular season games: {len(m_reg)}")
print(f"  Women's regular season games: {len(w_reg)}")
print(f"  Men's tourney games: {len(m_tourney)}")
print(f"  Women's tourney games: {len(w_tourney)}")
print(f"  Submission rows: {len(submission)}")

STEP 1: Loading data...
  Men's regular season games: 198079
  Women's regular season games: 142093
  Men's tourney games: 2585
  Women's tourney games: 1717
  Submission rows: 132133


In [6]:
# =============================================================================
# 2. ELO RATING SYSTEM
# =============================================================================
print("\n" + "=" * 60)
print("STEP 2: Building Elo ratings...")
print("=" * 60)

def build_elo_ratings(reg_season_df, K=20, HOME_ADV=100, MEAN_ELO=1500, REVERT=0.75):
    """
    Build Elo ratings by replaying all regular season games chronologically.
    Returns dict: {season: {team_id: elo}} — end-of-season ratings.
    """
    elo = {}  # current elo for each team
    season_elos = {}  # stored end-of-season snapshots

    seasons = sorted(reg_season_df['Season'].unique())

    for season in seasons:
        # Start-of-season reversion to mean
        for team in elo:
            elo[team] = MEAN_ELO + REVERT * (elo[team] - MEAN_ELO)

        season_games = reg_season_df[reg_season_df['Season'] == season].sort_values('DayNum')

        for _, game in season_games.iterrows():
            w = game['WTeamID']
            l = game['LTeamID']

            # Initialize new teams
            if w not in elo:
                elo[w] = MEAN_ELO
            if l not in elo:
                elo[l] = MEAN_ELO

            # Home advantage
            w_adv = HOME_ADV if game['WLoc'] == 'H' else (-HOME_ADV if game['WLoc'] == 'A' else 0)

            # Expected scores
            w_exp = 1.0 / (1.0 + 10 ** ((elo[l] - elo[w] - w_adv) / 400))

            # Margin of victory multiplier (diminishing returns)
            mov = game['WScore'] - game['LScore']
            mov_mult = np.log(max(mov, 1) + 1) * (2.2 / ((elo[w] - elo[l]) * 0.001 + 2.2))

            # Update
            update = K * mov_mult * (1 - w_exp)
            elo[w] += update
            elo[l] -= update

        # Store end-of-season snapshot
        season_elos[season] = dict(elo)

    return season_elos


m_elo = build_elo_ratings(m_reg)
w_elo = build_elo_ratings(w_reg)

# Quick sanity check: top teams in 2025
if 2025 in m_elo:
    top_2025 = sorted(m_elo[2025].items(), key=lambda x: -x[1])[:10]
    top_names = []
    for tid, rating in top_2025:
        name = m_teams[m_teams['TeamID'] == tid]['TeamName'].values
        top_names.append(f"  {name[0] if len(name) > 0 else tid}: {rating:.0f}")
    print("Top 10 Men's Elo (end of 2025 season):")
    print("\n".join(top_names))


STEP 2: Building Elo ratings...
Top 10 Men's Elo (end of 2025 season):
  Houston: 2026
  Florida: 1999
  Duke: 1983
  St John's: 1944
  Auburn: 1940
  Michigan St: 1917
  Tennessee: 1901
  Gonzaga: 1893
  St Mary's CA: 1889
  Drake: 1876


In [7]:
# =============================================================================
# 3. TEAM-LEVEL SEASON STATISTICS (Four Factors + extras)
# =============================================================================
print("\n" + "=" * 60)
print("STEP 3: Computing team-level season statistics...")
print("=" * 60)

def compute_team_stats(detail_df):
    """
    Compute per-team, per-season stats from detailed results.
    Returns a DataFrame indexed by (Season, TeamID).
    """
    records = []

    # Process winning team stats
    for prefix, team_col, opp_prefix in [('W', 'WTeamID', 'L'), ('L', 'LTeamID', 'W')]:
        df = detail_df.copy()
        df['TeamID'] = df[team_col]
        df['Score'] = df[f'{prefix}Score']
        df['OppScore'] = df[f'{opp_prefix}Score']

        for stat in ['FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA', 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF']:
            df[f'T_{stat}'] = df[f'{prefix}{stat}']
            df[f'O_{stat}'] = df[f'{opp_prefix}{stat}']

        df['Win'] = 1 if prefix == 'W' else 0
        records.append(df[['Season', 'TeamID', 'Score', 'OppScore', 'Win'] +
                         [f'T_{s}' for s in ['FGM','FGA','FGM3','FGA3','FTM','FTA','OR','DR','Ast','TO','Stl','Blk','PF']] +
                         [f'O_{s}' for s in ['FGM','FGA','FGM3','FGA3','FTM','FTA','OR','DR','Ast','TO','Stl','Blk','PF']]])

    all_games = pd.concat(records, ignore_index=True)

    # Aggregate to season-level
    agg = all_games.groupby(['Season', 'TeamID']).agg(
        Games=('Win', 'count'),
        Wins=('Win', 'sum'),
        PointsFor=('Score', 'mean'),
        PointsAgainst=('OppScore', 'mean'),
        # Offense
        T_FGM=('T_FGM', 'sum'), T_FGA=('T_FGA', 'sum'),
        T_FGM3=('T_FGM3', 'sum'), T_FGA3=('T_FGA3', 'sum'),
        T_FTM=('T_FTM', 'sum'), T_FTA=('T_FTA', 'sum'),
        T_OR=('T_OR', 'sum'), T_DR=('T_DR', 'sum'),
        T_Ast=('T_Ast', 'sum'), T_TO=('T_TO', 'sum'),
        T_Stl=('T_Stl', 'sum'), T_Blk=('T_Blk', 'sum'),
        # Defense (opponent stats)
        O_FGM=('O_FGM', 'sum'), O_FGA=('O_FGA', 'sum'),
        O_FGM3=('O_FGM3', 'sum'), O_FGA3=('O_FGA3', 'sum'),
        O_FTM=('O_FTM', 'sum'), O_FTA=('O_FTA', 'sum'),
        O_OR=('O_OR', 'sum'), O_DR=('O_DR', 'sum'),
        O_Ast=('O_Ast', 'sum'), O_TO=('O_TO', 'sum'),
        O_Stl=('O_Stl', 'sum'), O_Blk=('O_Blk', 'sum'),
    ).reset_index()

    # Derived metrics — Four Factors style
    # Possessions estimate
    agg['T_Poss'] = agg['T_FGA'] - agg['T_OR'] + agg['T_TO'] + 0.475 * agg['T_FTA']
    agg['O_Poss'] = agg['O_FGA'] - agg['O_OR'] + agg['O_TO'] + 0.475 * agg['O_FTA']
    agg['Poss'] = (agg['T_Poss'] + agg['O_Poss']) / 2

    # Tempo (possessions per game)
    agg['Tempo'] = agg['Poss'] / agg['Games']

    # Offensive efficiency (points per 100 possessions)
    agg['OE'] = agg['PointsFor'] / (agg['Poss'] / agg['Games']) * 100
    agg['DE'] = agg['PointsAgainst'] / (agg['Poss'] / agg['Games']) * 100
    agg['NetEff'] = agg['OE'] - agg['DE']

    # Four Factors — Offense
    agg['eFG_pct'] = (agg['T_FGM'] + 0.5 * agg['T_FGM3']) / agg['T_FGA']
    agg['TO_pct'] = agg['T_TO'] / agg['Poss']
    agg['OR_pct'] = agg['T_OR'] / (agg['T_OR'] + agg['O_DR'])
    agg['FTR'] = agg['T_FTM'] / agg['T_FGA']

    # Four Factors — Defense
    agg['D_eFG_pct'] = (agg['O_FGM'] + 0.5 * agg['O_FGM3']) / agg['O_FGA']
    agg['D_TO_pct'] = agg['O_TO'] / agg['Poss']
    agg['D_OR_pct'] = agg['O_OR'] / (agg['O_OR'] + agg['T_DR'])
    agg['D_FTR'] = agg['O_FTM'] / agg['O_FGA']

    # Additional useful stats
    agg['WinPct'] = agg['Wins'] / agg['Games']
    agg['AvgMargin'] = agg['PointsFor'] - agg['PointsAgainst']
    agg['Ast_TO'] = agg['T_Ast'] / np.maximum(agg['T_TO'], 1)
    agg['Stl_pct'] = agg['T_Stl'] / agg['Poss']
    agg['Blk_pct'] = agg['T_Blk'] / agg['O_FGA']
    agg['FG3_pct'] = agg['T_FGM3'] / np.maximum(agg['T_FGA3'], 1)
    agg['FT_pct'] = agg['T_FTM'] / np.maximum(agg['T_FTA'], 1)

    return agg

m_stats = compute_team_stats(m_detail)
w_stats = compute_team_stats(w_detail)

print(f"  Men's team-seasons with detailed stats: {len(m_stats)}")
print(f"  Women's team-seasons with detailed stats: {len(w_stats)}")


STEP 3: Computing team-level season statistics...
  Men's team-seasons with detailed stats: 8346
  Women's team-seasons with detailed stats: 5965


In [8]:
# =============================================================================
# 4. SEED-BASED FEATURES
# =============================================================================
print("\n" + "=" * 60)
print("STEP 4: Processing seed information...")
print("=" * 60)

def parse_seed(seed_str):
    """Extract numeric seed (1-16) from seed string like 'W01', 'X14a'."""
    return int(seed_str[1:3])

for df in [m_seeds, w_seeds]:
    df['SeedNum'] = df['Seed'].apply(parse_seed)

# Historical seed-vs-seed win rates
def compute_seed_win_rates(seeds_df, tourney_df):
    """Compute P(lower seed wins) for each seed matchup."""
    tourney = tourney_df.merge(
        seeds_df[['Season', 'TeamID', 'SeedNum']],
        left_on=['Season', 'WTeamID'], right_on=['Season', 'TeamID']
    ).rename(columns={'SeedNum': 'WSeed'}).drop('TeamID', axis=1)

    tourney = tourney.merge(
        seeds_df[['Season', 'TeamID', 'SeedNum']],
        left_on=['Season', 'LTeamID'], right_on=['Season', 'TeamID']
    ).rename(columns={'SeedNum': 'LSeed'}).drop('TeamID', axis=1)

    records = []
    for _, g in tourney.iterrows():
        s1, s2 = min(g['WSeed'], g['LSeed']), max(g['WSeed'], g['LSeed'])
        lower_seed_won = 1 if g['WSeed'] == s1 else 0
        records.append({'Seed1': s1, 'Seed2': s2, 'LowerWon': lower_seed_won})

    df = pd.DataFrame(records)
    rates = df.groupby(['Seed1', 'Seed2'])['LowerWon'].agg(['mean', 'count']).reset_index()
    rates.columns = ['Seed1', 'Seed2', 'WinRate', 'Count']
    return rates

m_seed_rates = compute_seed_win_rates(m_seeds, m_tourney)
w_seed_rates = compute_seed_win_rates(w_seeds, w_tourney)

print(f"  Men's seed matchup types: {len(m_seed_rates)}")
print("  Example: 1 vs 16 win rate:", 
      m_seed_rates[(m_seed_rates.Seed1==1) & (m_seed_rates.Seed2==16)]['WinRate'].values[0] if len(m_seed_rates[(m_seed_rates.Seed1==1) & (m_seed_rates.Seed2==16)]) > 0 else 'N/A')


STEP 4: Processing seed information...
  Men's seed matchup types: 84
  Example: 1 vs 16 win rate: 0.9875


In [9]:
# =============================================================================
# 5. FEATURE ENGINEERING FOR MATCHUPS
# =============================================================================
print("\n" + "=" * 60)
print("STEP 5: Building training dataset from historical tournaments...")
print("=" * 60)

FEATURE_COLS = [
    'Elo_diff', 'Seed_diff',
    'WinPct_diff', 'AvgMargin_diff',
    'OE_diff', 'DE_diff', 'NetEff_diff', 'Tempo_diff',
    'eFG_pct_diff', 'TO_pct_diff', 'OR_pct_diff', 'FTR_diff',
    'D_eFG_pct_diff', 'D_TO_pct_diff', 'D_OR_pct_diff', 'D_FTR_diff',
    'Ast_TO_diff', 'Stl_pct_diff', 'Blk_pct_diff',
    'FG3_pct_diff', 'FT_pct_diff',
    'SeedWinRate',
]

STAT_FEATURES = [
    'WinPct', 'AvgMargin', 'OE', 'DE', 'NetEff', 'Tempo',
    'eFG_pct', 'TO_pct', 'OR_pct', 'FTR',
    'D_eFG_pct', 'D_TO_pct', 'D_OR_pct', 'D_FTR',
    'Ast_TO', 'Stl_pct', 'Blk_pct', 'FG3_pct', 'FT_pct',
]


def build_training_data(tourney_df, seeds_df, elo_dict, stats_df, seed_rates_df, gender='M'):
    """
    Build training rows from historical tournament games.
    Each row: features = TeamA_stats - TeamB_stats (where A has lower TeamID).
    Target = 1 if lower TeamID won, 0 otherwise.
    """
    rows = []

    # Only use seasons where we have detailed stats
    available_seasons = stats_df['Season'].unique()

    for _, game in tourney_df.iterrows():
        season = game['Season']
        if season not in available_seasons:
            continue

        w_id = game['WTeamID']
        l_id = game['LTeamID']

        # Canonical ordering: lower TeamID is "TeamA"
        if w_id < l_id:
            team_a, team_b = w_id, l_id
            target = 1  # lower ID won
        else:
            team_a, team_b = l_id, w_id
            target = 0  # lower ID lost

        row = {'Season': season, 'TeamA': team_a, 'TeamB': team_b, 'Target': target}

        # Elo
        elo_season = elo_dict.get(season, {})
        elo_a = elo_season.get(team_a, 1500)
        elo_b = elo_season.get(team_b, 1500)
        row['Elo_diff'] = elo_a - elo_b

        # Seeds
        seed_a_row = seeds_df[(seeds_df['Season'] == season) & (seeds_df['TeamID'] == team_a)]
        seed_b_row = seeds_df[(seeds_df['Season'] == season) & (seeds_df['TeamID'] == team_b)]
        seed_a = seed_a_row['SeedNum'].values[0] if len(seed_a_row) > 0 else 8.5
        seed_b = seed_b_row['SeedNum'].values[0] if len(seed_b_row) > 0 else 8.5
        row['Seed_diff'] = seed_b - seed_a  # positive = team_a has better (lower) seed

        # Seed win rate lookup
        s1, s2 = int(min(seed_a, seed_b)), int(max(seed_a, seed_b))
        sr = seed_rates_df[(seed_rates_df['Seed1'] == s1) & (seed_rates_df['Seed2'] == s2)]
        if len(sr) > 0:
            seed_wr = sr['WinRate'].values[0]
            # Flip if team_a doesn't have the lower seed
            row['SeedWinRate'] = seed_wr if seed_a <= seed_b else (1 - seed_wr)
        else:
            row['SeedWinRate'] = 0.5

        # Team stats differences
        stats_a = stats_df[(stats_df['Season'] == season) & (stats_df['TeamID'] == team_a)]
        stats_b = stats_df[(stats_df['Season'] == season) & (stats_df['TeamID'] == team_b)]

        if len(stats_a) == 0 or len(stats_b) == 0:
            continue

        for feat in STAT_FEATURES:
            val_a = stats_a[feat].values[0]
            val_b = stats_b[feat].values[0]
            row[f'{feat}_diff'] = val_a - val_b

        rows.append(row)

    return pd.DataFrame(rows)


m_train = build_training_data(m_tourney, m_seeds, m_elo, m_stats, m_seed_rates, 'M')
w_train = build_training_data(w_tourney, w_seeds, w_elo, w_stats, w_seed_rates, 'W')

print(f"  Men's training rows: {len(m_train)}")
print(f"  Women's training rows: {len(w_train)}")
print(f"  Men's target distribution: {m_train['Target'].mean():.3f}")
print(f"  Women's target distribution: {w_train['Target'].mean():.3f}")


STEP 5: Building training dataset from historical tournaments...
  Men's training rows: 1449
  Women's training rows: 961
  Men's target distribution: 0.500
  Women's target distribution: 0.505


In [10]:
# =============================================================================
# 6. MODEL TRAINING
# =============================================================================
print("\n" + "=" * 60)
print("STEP 6: Training models...")
print("=" * 60)

def train_ensemble(train_df, feature_cols, label=''):
    """Train logistic regression + gradient boosting, return fitted models."""
    X = train_df[feature_cols].copy()
    y = train_df['Target'].values

    # Fill NaN/inf
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

    # Standardize for logistic regression
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Model 1: Logistic Regression (calibrated)
    lr = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs')
    lr_scores = cross_val_score(lr, X_scaled, y, cv=5, scoring='neg_brier_score')
    print(f"  [{label}] Logistic Regression CV Brier: {-lr_scores.mean():.4f} (+/- {lr_scores.std():.4f})")
    lr.fit(X_scaled, y)

    # Model 2: HistGradientBoosting
    hgb = HistGradientBoostingClassifier(
        max_iter=300, max_depth=4, learning_rate=0.05,
        min_samples_leaf=20, l2_regularization=1.0,
        random_state=42
    )
    hgb_scores = cross_val_score(hgb, X, y, cv=5, scoring='neg_brier_score')
    print(f"  [{label}] HistGBM CV Brier: {-hgb_scores.mean():.4f} (+/- {hgb_scores.std():.4f})")
    hgb.fit(X, y)

    # Model 3: Calibrated HistGBM
    hgb_cal = CalibratedClassifierCV(
        HistGradientBoostingClassifier(
            max_iter=300, max_depth=4, learning_rate=0.05,
            min_samples_leaf=20, l2_regularization=1.0,
            random_state=42
        ),
        cv=5, method='isotonic'
    )
    hgb_cal.fit(X, y)

    return lr, hgb, hgb_cal, scaler


m_lr, m_hgb, m_hgb_cal, m_scaler = train_ensemble(m_train, FEATURE_COLS, 'Men')
w_lr, w_hgb, w_hgb_cal, w_scaler = train_ensemble(w_train, FEATURE_COLS, 'Women')

# Feature importance (from HistGBM)
print("\n  Men's Top Feature Importances (Logistic Regression coefficients):")
importances = pd.Series(np.abs(m_lr.coef_[0]), index=FEATURE_COLS).sort_values(ascending=False)
for feat, imp in importances.head(10).items():
    print(f"    {feat}: {imp:.3f}")


STEP 6: Training models...
  [Men] Logistic Regression CV Brier: 0.1855 (+/- 0.0119)


  File "C:\Users\T0291175\MyApp\VirtualEnvs\ml_env\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Program Files\Python312\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\Python312\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Program Files\Python312\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


  [Men] HistGBM CV Brier: 0.1982 (+/- 0.0155)
  [Women] Logistic Regression CV Brier: 0.1419 (+/- 0.0088)
  [Women] HistGBM CV Brier: 0.1617 (+/- 0.0072)

  Men's Top Feature Importances (Logistic Regression coefficients):
    Elo_diff: 0.788
    SeedWinRate: 0.502
    AvgMargin_diff: 0.476
    D_TO_pct_diff: 0.442
    WinPct_diff: 0.418
    DE_diff: 0.412
    Ast_TO_diff: 0.263
    D_eFG_pct_diff: 0.251
    NetEff_diff: 0.220
    TO_pct_diff: 0.220


In [13]:
# =============================================================================
# 7. GENERATE 2026 PREDICTIONS
# =============================================================================
print("\n" + "=" * 60)
print("STEP 7: Generating 2026 predictions...")
print("=" * 60)

# Parse submission IDs
submission['Season'] = submission['ID'].apply(lambda x: int(x.split('_')[0]))
submission['TeamA'] = submission['ID'].apply(lambda x: int(x.split('_')[1]))
submission['TeamB'] = submission['ID'].apply(lambda x: int(x.split('_')[2]))

# Split men's vs women's
sub_m = submission[submission['TeamA'] < 2000].copy()
sub_w = submission[submission['TeamA'] >= 3000].copy()
print(f"  Men's matchups to predict: {len(sub_m)}")
print(f"  Women's matchups to predict: {len(sub_w)}")


def generate_predictions(sub_df, elo_dict, stats_df, seeds_df, seed_rates_df,
                         lr_model, hgb_model, hgb_cal_model, scaler, 
                         feature_cols, season=2026):
    """Generate predictions for all matchups in submission."""

    elo_season = elo_dict.get(season, {})
    seeds_season = seeds_df[seeds_df['Season'] == season]
    stats_season = stats_df[stats_df['Season'] == season]

    # Build seed lookup
    seed_lookup = {}
    for _, row in seeds_season.iterrows():
        seed_lookup[row['TeamID']] = row['SeedNum']

    # Build stats lookup
    stats_lookup = {}
    for _, row in stats_season.iterrows():
        stats_lookup[row['TeamID']] = row

    rows = []
    for _, g in sub_df.iterrows():
        team_a, team_b = g['TeamA'], g['TeamB']
        row = {}

        # Elo
        elo_a = elo_season.get(team_a, 1500)
        elo_b = elo_season.get(team_b, 1500)
        row['Elo_diff'] = elo_a - elo_b

        # Seeds
        seed_a = seed_lookup.get(team_a, 8.5)
        seed_b = seed_lookup.get(team_b, 8.5)
        row['Seed_diff'] = seed_b - seed_a

        # Seed win rate
        s1, s2 = int(min(seed_a, seed_b)), int(max(seed_a, seed_b))
        sr = seed_rates_df[(seed_rates_df['Seed1'] == s1) & (seed_rates_df['Seed2'] == s2)]
        if len(sr) > 0:
            seed_wr = sr['WinRate'].values[0]
            row['SeedWinRate'] = seed_wr if seed_a <= seed_b else (1 - seed_wr)
        else:
            row['SeedWinRate'] = 0.5

        # Stats diff
        sa = stats_lookup.get(team_a)
        sb = stats_lookup.get(team_b)
        for feat in STAT_FEATURES:
            val_a = sa[feat] if sa is not None else 0
            val_b = sb[feat] if sb is not None else 0
            row[f'{feat}_diff'] = val_a - val_b

        rows.append(row)

    X_pred = pd.DataFrame(rows)[feature_cols]
    X_pred = X_pred.replace([np.inf, -np.inf], np.nan).fillna(0)

    # Predictions from each model
    X_scaled = scaler.transform(X_pred)
    p_lr = lr_model.predict_proba(X_scaled)[:, 1]
    p_hgb = hgb_model.predict_proba(X_pred)[:, 1]
    p_hgb_cal = hgb_cal_model.predict_proba(X_pred)[:, 1]

    # Ensemble: weighted average
    # Give more weight to calibrated GBM + logistic
    preds = 0.30 * p_lr + 0.30 * p_hgb + 0.40 * p_hgb_cal

    # Clip to avoid extreme probabilities
    preds = np.clip(preds, 0.02, 0.98)

    return preds


sub_m['Pred'] = generate_predictions(
    sub_m, m_elo, m_stats, m_seeds, m_seed_rates,
    m_lr, m_hgb, m_hgb_cal, m_scaler, FEATURE_COLS
)

sub_w['Pred'] = generate_predictions(
    sub_w, w_elo, w_stats, w_seeds, w_seed_rates,
    w_lr, w_hgb, w_hgb_cal, w_scaler, FEATURE_COLS
)

# Combine and format
final = pd.concat([sub_m, sub_w])[['ID', 'Pred']]
final = final.sort_values('ID').reset_index(drop=True)

# Sanity checks
print(f"\n  Prediction statistics:")
print(f"    Total predictions: {len(final)}")
print(f"    Mean: {final['Pred'].mean():.4f}")
print(f"    Std:  {final['Pred'].std():.4f}")
print(f"    Min:  {final['Pred'].min():.4f}")
print(f"    Max:  {final['Pred'].max():.4f}")
print(f"    Preds < 0.1: {(final['Pred'] < 0.1).sum()}")
print(f"    Preds > 0.9: {(final['Pred'] > 0.9).sum()}")

# Save
output_path = './outputs/submission.csv'
final.to_csv(output_path, index=False)
print(f"\n  Submission saved to {output_path}")


STEP 7: Generating 2026 predictions...
  Men's matchups to predict: 66430
  Women's matchups to predict: 65703

  Prediction statistics:
    Total predictions: 132133
    Mean: 0.5185
    Std:  0.2276
    Min:  0.0405
    Max:  0.9800
    Preds < 0.1: 872
    Preds > 0.9: 4439

  Submission saved to ./outputs/submission.csv


In [14]:
# =============================================================================
# 8. BACKTESTING ON RECENT TOURNAMENTS
# =============================================================================
print("\n" + "=" * 60)
print("STEP 8: Backtesting on 2021-2025 tournaments...")
print("=" * 60)

def backtest(tourney_df, seeds_df, elo_dict, stats_df, seed_rates_df,
             lr_class, hgb_class, feature_cols, stat_features, label=''):
    """Leave-one-season-out backtest on recent tournaments."""
    
    test_seasons = [s for s in range(2021, 2026) if s in stats_df['Season'].unique()]
    all_seasons_in_data = stats_df['Season'].unique()
    
    results = []
    for test_season in test_seasons:
        # Build training data from all OTHER seasons
        train_seasons = [s for s in all_seasons_in_data if s != test_season and s <= test_season]
        tourney_train = tourney_df[tourney_df['Season'].isin(train_seasons)]
        
        train_data = build_training_data(tourney_train, seeds_df, elo_dict, stats_df, seed_rates_df)
        if len(train_data) < 50:
            continue
            
        # Build test data
        tourney_test = tourney_df[tourney_df['Season'] == test_season]
        test_data = build_training_data(tourney_test, seeds_df, elo_dict, stats_df, seed_rates_df)
        if len(test_data) == 0:
            continue
        
        X_train = train_data[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
        y_train = train_data['Target'].values
        X_test = test_data[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
        y_test = test_data['Target'].values
        
        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)
        
        # LR
        lr = LogisticRegression(C=1.0, max_iter=1000)
        lr.fit(X_train_s, y_train)
        p_lr = lr.predict_proba(X_test_s)[:, 1]
        
        # HGB
        hgb = HistGradientBoostingClassifier(max_iter=300, max_depth=4, learning_rate=0.05,
                                              min_samples_leaf=20, l2_regularization=1.0, random_state=42)
        hgb.fit(X_train, y_train)
        p_hgb = hgb.predict_proba(X_test)[:, 1]
        
        # Ensemble
        p_ens = 0.4 * p_lr + 0.6 * p_hgb
        p_ens = np.clip(p_ens, 0.02, 0.98)
        
        brier = np.mean((p_ens - y_test) ** 2)
        accuracy = np.mean((p_ens > 0.5) == y_test)
        results.append({'Season': test_season, 'Brier': brier, 'Accuracy': accuracy, 'Games': len(y_test)})
        
    if results:
        res_df = pd.DataFrame(results)
        for _, r in res_df.iterrows():
            print(f"  [{label}] {int(r['Season'])}: Brier={r['Brier']:.4f}, Accuracy={r['Accuracy']:.1%}, Games={int(r['Games'])}")
        print(f"  [{label}] Average Brier: {res_df['Brier'].mean():.4f}")

backtest(m_tourney, m_seeds, m_elo, m_stats, m_seed_rates,
         LogisticRegression, HistGradientBoostingClassifier, FEATURE_COLS, STAT_FEATURES, 'Men')
backtest(w_tourney, w_seeds, w_elo, w_stats, w_seed_rates,
         LogisticRegression, HistGradientBoostingClassifier, FEATURE_COLS, STAT_FEATURES, 'Women')

print("\n" + "=" * 60)
print("PIPELINE COMPLETE!")
print("=" * 60)


STEP 8: Backtesting on 2021-2025 tournaments...
  [Men] 2021: Brier=0.2051, Accuracy=68.2%, Games=66
  [Men] 2022: Brier=0.2216, Accuracy=67.2%, Games=67
  [Men] 2023: Brier=0.2127, Accuracy=70.1%, Games=67
  [Men] 2024: Brier=0.1839, Accuracy=71.6%, Games=67
  [Men] 2025: Brier=0.1568, Accuracy=77.6%, Games=67
  [Men] Average Brier: 0.1960
  [Women] 2021: Brier=0.1681, Accuracy=71.4%, Games=63
  [Women] 2022: Brier=0.1740, Accuracy=73.1%, Games=67
  [Women] 2023: Brier=0.2063, Accuracy=70.1%, Games=67
  [Women] 2024: Brier=0.1375, Accuracy=82.1%, Games=67
  [Women] 2025: Brier=0.0865, Accuracy=88.1%, Games=67
  [Women] Average Brier: 0.1545

PIPELINE COMPLETE!
